In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], "figure.dpi": 110})

SLIM_DIR = "/Volumes/Lexar/mglurs-brain-sc-datasets/derived/regions"
FIG_DIR = "figures/allen_wmb"
RES_DIR = "results/allen_wmb"
NAMES = {'ACA': 'Anterior cingulate area', 'AI': 'Agranular insular area', 'AUD': 'Auditory areas', 'AUD-TEa-PERI-ECT': 'Auditory/temporal/perirhinal/ectorhinal areas', 'CB': 'Cerebellum', 'CTXsp': 'Cortical subplate', 'ENT': 'Entorhinal area', 'HIP': 'Hippocampal region', 'HY': 'Hypothalamus', 'LSX': 'Lateral septal complex', 'MB': 'Midbrain', 'MY': 'Medulla', 'OLF': 'Olfactory areas', 'P': 'Pons', 'PAL': 'Pallidum', 'RHP': 'Retrohippocampal region', 'RSP': 'Retrosplenial area', 'SSp': 'Primary somatosensory area', 'STRd': 'Striatum dorsal region', 'STRv': 'Striatum ventral region', 'TH': 'Thalamus', 'VIS': 'Visual areas', 'sAMY': 'Striatum-like amygdalar nuclei', 'MOp': 'Primary motor area', 'VIS-PTLp': 'Visual/posterior parietal  areas', 'TEa-PERI-ECT': 'Temporal/perirhinal/ectorhinal areas ', 'PL-ILA-ORB': 'Prelimbic/infralimbic/orbital areas ', 'SS-GU-VISC': 'Somatosensory/gustatory/visceral areas', 'MO-FRP': 'Somatomotor - Frontal pole'}
MIN_CELLS = 200
GRMS = [f"Grm{i}" for i in range(1, 9)]

_t20 = plt.get_cmap("tab20").colors
_all = [_t20[i] for i in range(0, 20, 2)] + [_t20[i] for i in range(1, 20, 2)] + list(plt.get_cmap("tab20b").colors)
PAL = [c for c in _all if not (abs(c[0] - c[1]) < 0.03 and abs(c[1] - c[2]) < 0.03)]

def class_num(c):
    n = pd.to_numeric(str(c)[:2], errors="coerce")
    return int(n) if pd.notna(n) else 0

def class_color(c):
    n = class_num(c)
    return PAL[(n - 1) % len(PAL)] if n > 0 else (0.5, 0.5, 0.5)

def is_neuron_class(c):
    return not (30 <= class_num(c) <= 34)

def neuron_type(c):
    c = str(c)
    if not is_neuron_class(c):
        return "non-neuronal"
    if "GABA" in c:
        return "GABAergic"
    if "Glut" in c:
        return "glutamatergic"
    return "other neuronal"

FOCUS_ROIS = ["PL-ILA-ORB", "ACA", "HIP", "CTXsp", "sAMY", "STRd", "STRv"]
PAIRS = [("Grm7", "Grm8"), ("Grm7", "Grm2"), ("Grm7", "Grm3"), ("Grm2", "Grm4")]
GROUP_GENES = {"Grm": ["Grm1", "Grm2", "Grm3", "Grm4", "Grm5", "Grm6", "Grm7", "Grm8"],
               "housekeeping": ["Gapdh", "Actb", "Eef1a1", "Tubb3", "Tuba1a", "Rpl13a", "B2m"],
               "other GPCRs": ["Gabbr1", "Gabbr2", "Gpr158", "Gprc5b", "Drd1", "Drd2", "Htr2a", "Chrm1", "Adora1", "Cnr1"],
               "long neuronal": ["Csmd1", "Nrxn1", "Nrxn3", "Cntnap2", "Ctnna2", "Ptprd", "Dlg2", "Auts2"]}

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

from scipy.stats import spearmanr

res = pd.read_csv(f"{RES_DIR}/master_enrichment.csv.gz")
r = res[res["chem"].isin(["WMB-10Xv2", "WMB-10Xv3"])]
w = r.pivot_table(index=["roi", "gene", "class", "subclass"], columns="chem", values="fold").dropna().reset_index()
w["lfc_v2"] = np.log2(w["WMB-10Xv2"])
w["lfc_v3"] = np.log2(w["WMB-10Xv3"])
genes = [g for g in GRMS if g in set(w["gene"])]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
out = []
for ax, g in zip(axes.ravel(), genes):
    x = w[w["gene"] == g]
    rho = spearmanr(x["lfc_v2"], x["lfc_v3"])[0] if len(x) > 2 else np.nan
    ax.scatter(x["lfc_v2"], x["lfc_v3"], s=6, c=[class_color(c) for c in x["class"]], linewidths=0, alpha=0.8)
    lim = [min(x["lfc_v2"].min(), x["lfc_v3"].min()), max(x["lfc_v2"].max(), x["lfc_v3"].max())]
    ax.plot(lim, lim, color="k", lw=0.6, ls="--")
    ax.set_title(f"{g}: Spearman {rho:.2f}, n = {len(x)}", fontsize=10)
    ax.set_xlabel("log2 fold vs other neurons (10x v2)", fontsize=8)
    ax.set_ylabel("log2 fold (10x v3)", fontsize=8)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    out.append((g, rho, len(x)))
for ax in axes.ravel()[len(genes):]:
    ax.axis("off")
fig.suptitle("Replicate check: enrichment of neuronal subclasses vs other neurons in the same region, 10x v2 vs v3\neach point = one subclass in one region; fold = (mean CPM in subclass + 1) / (mean CPM in other neurons + 1); color = class", fontsize=11)
fig.savefig(f"{FIG_DIR}/ALL_fig5_v2_v3_concordance.png", dpi=250, bbox_inches="tight")
pd.DataFrame(out, columns=["gene", "spearman", "n_pairs"]).to_csv(f"{RES_DIR}/v2_v3_concordance.csv", index=False)
by = []
for (roi, g), x in w.groupby(["roi", "gene"]):
    if len(x) >= 8:
        by.append((roi, g, spearmanr(x["lfc_v2"], x["lfc_v3"])[0], len(x)))
pd.DataFrame(by, columns=["roi", "gene", "spearman", "n_pairs"]).to_csv(f"{RES_DIR}/v2_v3_concordance_by_region.csv", index=False)
print(pd.DataFrame(out, columns=["gene", "spearman", "n_pairs"]).round(2).to_string(index=False))
plt.show()